# Activations Manager

This notebook includes cells to perform different kinds of data manipulation with the activations

## Import libraries

In [ ]:
from activations import *

from utils import ACTIVATIONS_FOLDER, MODELS_FOLDER, LANGUAGES, SPLITS, MODEL_NAMES
from icecream import ic

device: t.device = t.device("cuda" if t.cuda.is_available() else "cpu")
device

## Activation Generator

In [ ]:
save_to_disk: bool = True
amount_of_batches_to_generate: int | None = None
batch_size: int = 128
# model_name: str = "tiny_aya_global"
languages_to_generate: list[str] = LANGUAGES
splits_to_generate: list[str] = SPLITS

debug = False
if debug:
    languages_to_generate = ["en"]
    languages_to_generate = LANGUAGES
    splits_to_generate = ["train"]
    amount_of_batches_to_generate = 2
    batch_size = 4
    print("Running in debug mode")

ic(save_to_disk, amount_of_batches_to_generate, batch_size, languages_to_generate, splits_to_generate)


assert save_to_disk, ("If generating, must also save to disk")

In [ ]:
languages_to_generate = ["jp"]
batch_size = 64

model_name: str = "olmo_model"

activation_recorder: ActivationRecorder = ActivationRecorder(model_name)

special_cases: list[SpecialCase] = []    

activation_recorder.generate_all_activations(
    languages_to_generate,
    splits_to_generate,
    amount_of_batches_to_generate,
    save_to_disk,
    batch_size,
    special_cases=special_cases,
)

In [ ]:
languages_to_generate = LANGUAGES

model_name: str = "tiny_aya_global"

activation_recorder: ActivationRecorder = ActivationRecorder(model_name)

special_cases: list[SpecialCase] = []    

activation_recorder.generate_all_activations(
    languages_to_generate,
    splits_to_generate,
    amount_of_batches_to_generate,
    save_to_disk,
    batch_size,
    special_cases=special_cases,
)

## Sanity Checks

In [ ]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = True
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    splits = ["train"]
    print(f"Using custom configuration")
    ic(model_names, languages, splits)

for model_name in model_names:
    for language in languages:
        for split in splits:
            activations_dataset: ActivationDataset = ActivationDataset(language, split, 0, "standard", model_name)

            print(activations_dataset.activations.shape)
            for i in range(8):
                print(activations_dataset[i][0])
                print(activations_dataset[i][1])
                print("\n-----------------")

## Activation Merger

In [ ]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = False
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    splits = ["train", "test"]
    print("Using custom configuration")
else:
    print("Using default configuration")

ic(model_names, languages, splits, custom)

for model_name in model_names:
    num_layers: int = ActivationRecorder(model_name).get_number_of_layers()
    for language in languages:
        for split in splits:
            try:
                for layer_num in range(num_layers):
                    print(f"Merging {model_name}/{language}/{split}")
                    merge_activation_batches(model_name, language, split, layer_num)
            except FileNotFoundError as e:
                print(f"Couldn't merge activations of {model_name}, {language}, {split} due to:\n {e}")
                print("It is possible that these activations were already merged")

## Activation Deleter

In [ ]:
# models_to_delete: list[str] = ["olmo_model"]
# languages_to_delete: list[str] = ["es"]
models_to_delete: list[str] = MODEL_NAMES
languages_to_delete: list[str] = LANGUAGES
splits_to_delete: list[str] = SPLITS

actually_delete = True

for model_name in models_to_delete:
    for language in languages_to_delete:
        for split in splits_to_delete:
            delete_activations_file(model_name, language, split, layer_num=None, batch_id=None, ignore_substring="batch", actually_delete=actually_delete)


In [ ]:
models_to_delete: list[str] = MODEL_NAMES
languages_to_delete: list[str] = LANGUAGES
splits_to_delete: list[str] = SPLITS

actually_delete = False

for model_name in models_to_delete:
    for language in languages_to_delete:
        for split in splits_to_delete:
            delete_activations_file(model_name, language, split, layer_num=None, batch_id=None, ignore_substring="merged", actually_delete=actually_delete)